In [11]:
from pyserini.search.lucene import LuceneSearcher
import json
from tqdm import tqdm
import numpy as np
import os

In [6]:
with open("data/id_to_corpus_dict.json", "r") as f:
    id_to_corpus = json.load(f)
with open("data/id_to_query_dict.json", "r") as f:
    id_to_query = json.load(f)
with open("data/test_qrels_dict.json", "r") as f:
    test_qrels = json.load(f)

In [7]:
corpus_ids = list(id_to_corpus.keys())
query_ids = list(test_qrels.keys())

In [2]:
searcher = LuceneSearcher('quora_trec_index')

In [9]:
recalls = []
successes = []
for i in tqdm(range(len(test_qrels))):
    rel_docs = test_qrels[query_ids[i]]
    query = id_to_query[query_ids[i]]
    # doc_mat_ids = topk_doc_ids[i]
    retrieved_doc_ids = [hit.docid for hit in searcher.search(query, k=100)]
    intersection = list(set(rel_docs).intersection(set(retrieved_doc_ids)))
    recall = len(intersection) / len(rel_docs)
    success = int(len(intersection) > 0)
    recalls.append(recall)
    successes.append(success)

100%|██████████| 5008/5008 [00:22<00:00, 222.78it/s]


In [ ]:
results_path = f"models/baseline/"
os.makedirs(results_path, exist_ok=True)
with open(results_path + f"/results.txt", "w") as f:
    f.write(f"Recall@100 Quora: {np.mean(recalls[:-8])}\n")
    f.write(f"Success@100 Quora: {np.mean(successes[:-8])}\n")
    f.write(f"Recall@100 Trec: {np.mean(recalls[-8:])}\n")
    f.write(f"Success@100 Trec: {np.mean(successes[-8:])}\n")

: 